# dataloader-batching — worked example 2: Stack features and labels into batched shapes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dataloader-batching`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When a `TensorDataset` holds two aligned tensors — a 2-D feature matrix `(N, D)` and a 1-D label vector `(N,)` — the `DataLoader` stacks each along a new leading batch axis. A batch of size `B` yields features `(B, D)` and labels `(B,)`. The collation is automatic via the default `collate_fn`, which calls `torch.stack` per dataset column.

## Worked solution

**Goal:** show exactly what shapes come out when iterating a two-tensor dataset.

1. Re-seed with `t.manual_seed(0)` then draw `x = t.randn(N, D)` features and integer labels `y = t.randint(0, 3, (N,))`. Seeding inside the function makes the draw reproducible under the grader.
2. `TensorDataset(x, y)` pairs row `i` of `x` with element `i` of `y`. Indexing the dataset returns a tuple `(x_row, y_scalar)`.
3. `DataLoader(ds, batch_size=B, shuffle=False)` groups consecutive rows. The default collate stacks the `B` feature rows into `(B, D)` and the `B` label scalars into `(B,)`.
4. We grab the first batch with `next(iter(loader))`, unpack it as `(xb, yb)`, and read the shapes. With `shuffle=False` and `N` not divisible by `B`, the *first* batch is full size `B`; only the last batch could be smaller.
5. We also confirm dtype is preserved: features stay `float32`, labels stay `int64` (`torch.long`) — stacking never casts.

**Why it works:** the default collation treats each dataset column independently and applies `torch.stack`, which adds a new dim 0 of length `B` while leaving the per-example trailing shape untouched.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader


def first_batch_shapes(N, D, B):
    t.manual_seed(0)
    x = t.randn(N, D)
    y = t.randint(0, 3, (N,))
    ds = TensorDataset(x, y)
    loader = DataLoader(ds, batch_size=B, shuffle=False)
    xb, yb = next(iter(loader))
    return xb, yb


xb, yb = first_batch_shapes(17, 4, 6)
print('features:', tuple(xb.shape), xb.dtype)
print('labels:', tuple(yb.shape), yb.dtype)